# NVDA HTTP Bridge Endpoint Tests

This notebook tests the running NVDA HTTP bridge endpoints on `127.0.0.1:8765`. It sends requests to `/health`, `/tools`, and `/action` and validates the responses.

In [ ]:
import json
import urllib.request
from urllib.error import HTTPError, URLError
from urllib.parse import urljoin

BASE_URL = "http://127.0.0.1:8765"
TOKEN_HEADER = "X-NVDA-Bridge-Token"
BRIDGE_TOKEN = "nvda-bridge-secret-change-me"
HEADERS = {
    TOKEN_HEADER: BRIDGE_TOKEN,
    "Content-Type": "application/json",
}


In [ ]:
def send_request(path, method="GET", data=None):
    url = urljoin(BASE_URL, path)
    request_data = None
    if data is not None:
        request_data = json.dumps(data).encode("utf-8")
    req = urllib.request.Request(url, data=request_data, method=method)
    for key, value in HEADERS.items():
        req.add_header(key, value)

    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            body = resp.read().decode("utf-8")
            return resp.status, json.loads(body)
    except HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        return e.code, {"error": body}
    except URLError as e:
        return None, {"error": str(e)}


In [ ]:
test_paths = ["/health", "/tools"]
results = []
for path in test_paths:
    status, body = send_request(path)
    results.append((path, status, body))

# Test a tool action if the bridge responds successfully.
action_status, action_body = send_request(
    "/action",
    method="POST",
    data={"request_id": "http-test-1", "action": "get_current_focus", "params": {}},
)
results.append(("/action", action_status, action_body))

results

In [ ]:
for path, status, body in results:
    print(f"Request: {path}")
    print(f"  Status: {status}")
    print(f"  Body: {json.dumps(body, indent=2) if isinstance(body, dict) else body}")
    print("---")

success = all(isinstance(status, int) and status in (200, 201) for _, status, _ in results)
print("Bridge endpoint test result:", "PASS" if success else "FAIL")
